In [1]:
import json
import os
import numpy as np
from tqdm import tqdm
import math

def get_recommended_max_length(file_path, tokenizer, repeat_mode=False):
    """
    输入: json/jsonl文件路径, tokenizer
    输出: 推荐的 max_length (考虑了99%覆盖率和硬件对齐)
    """
    if not os.path.exists(file_path):
        print(f"错误: 文件 {file_path} 不存在")
        return 512

    # 1. 加载数据
    # 这里复用你之前的逻辑，确保格式一致
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read().strip()
        try:
            full_data = json.loads(content)
            data = full_data if isinstance(full_data, list) else [full_data]
        except json.JSONDecodeError:
            f.seek(0)
            for line in f:
                if line.strip(): data.append(json.loads(line))

    # 2. 统计长度
    lengths = []
    print(f"正在分析文件: {file_path} (模式: {'Repeat' if repeat_mode else 'Normal'})")
    
    for ex in tqdm(data):
        # 调用你代码里的 build_prompts 确保逻辑完全同步
        # 注意：这里需要传入 tokenizer 以应用 chat_template
        prompt = build_prompts(ex, tokenizer=tokenizer, repeat=repeat_mode)
        
        # 编码并计算长度
        tokens = tokenizer.encode(prompt, add_special_tokens=False)
        lengths.append(len(tokens))

    lengths = np.array(lengths)
    
    # 3. 计算关键统计指标
    max_len = int(np.max(lengths))
    p95 = int(np.percentile(lengths, 95))
    p99 = int(np.percentile(lengths, 99))
    
    # 4. 推荐逻辑：取99分位数，并向上取整到128的倍数（有利于GPU Tensor Core对齐）
    # 如果 p99 和 max 差距极小，建议直接用 max
    base_val = p99 if (max_len - p99) > 100 else max_len
    recommended = math.ceil(base_val / 128) * 128

    print("\n" + "="*30)
    print(f"统计结果 (单位: Tokens):")
    print(f"  平均长度: {np.mean(lengths):.1f}")
    print(f"  95% 覆盖: {p95}")
    print(f"  99% 覆盖: {p99}")
    print(f"  最大长度: {max_len}")
    print(f"  建议值 (Max_Length): {recommended}")
    print("="*30)
    
    return recommended

# 使用方法示例：
# from transformers import AutoTokenizer
# tokenizer = AutoTokenizer.from_pretrained("your_model_path")
# final_max_len = get_recommended_max_length("test_data.json", tokenizer, repeat_mode=True)

In [2]:
def _format_options_from_ex(ex):
    opt_obj = ex.get("options", [])
    if isinstance(opt_obj, list):
        return "Options:\n" + "\n".join(opt_obj)
    if isinstance(opt_obj, dict):
        return "Options:\n" + "\n".join([f"{k}) {v}" for k, v in opt_obj.items()])
    return ""

In [3]:

ASSISTANT_PROMPT = (
    "You are a logical task solver. Read the context, question and options carefully. "
    "First, provide a step-by-step reasoning chain to solve the problem. "
    "Finally, conclude your answer by strictly outputting the single option letter "
    "enclosed in LaTeX box format, for example: \\boxed{A}."
)

In [4]:
def build_prompts(ex, tokenizer=None, repeat=False, reverse_context=False):
    """
    构建 Prompt。如果 repeat=True，则应用论文中的 Query + Query 策略。
    """
    ctx = ex.get("context", "")
    q = ex.get("question", "")
    opts = _format_options_from_ex(ex)
    
    tail_prompt = "Please provide the reasoning and the answer."
    base_query = f"Context:\n{ctx}\n\nQuestion:\n{q}\n\n{opts}\n\n"
    if reverse_context:
        base_query = f"Question:\n{q}\n\n{opts}\n\nContext:\n{ctx}\n\n"
    
    # 核心：复现论文的 Prompt Repetition
    if repeat:
        # 你也可以在这里尝试论文里的变体：base_query + "\n\nLet me repeat that:\n\n" + base_query
        user_content = base_query + base_query + tail_prompt
    else:
        user_content = base_query + tail_prompt
    
    if tokenizer and hasattr(tokenizer, "apply_chat_template"):
        try:
            return tokenizer.apply_chat_template([
                {"role": "system", "content": ASSISTANT_PROMPT},
                {"role": "user", "content": user_content}
            ], tokenize=False, add_generation_prompt=True)
        except:
            return f"{ASSISTANT_PROMPT}\n\n{user_content}"
    return user_content

In [5]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("/data_a100/models/Qwen2.5-3B-Instruct", trust_remote_code=True)

In [6]:
test_data = ["data/AR-LSAT/test.json", "data/FOLIO/dev.json", "data/LogicalDeduction/dev.json","data/ProntoQA/dev.json", "data/ProofWriter/test.json"]  # 替换为你的测试数据路径

In [7]:
for data_path in test_data:
    final_max_len = get_recommended_max_length(data_path, tokenizer, repeat_mode=True)


正在分析文件: data/AR-LSAT/test.json (模式: Repeat)


100%|██████████| 230/230 [00:00<00:00, 972.90it/s]



统计结果 (单位: Tokens):
  平均长度: 518.2
  95% 覆盖: 671
  99% 覆盖: 793
  最大长度: 958
  建议值 (Max_Length): 896
正在分析文件: data/FOLIO/dev.json (模式: Repeat)


100%|██████████| 204/204 [00:00<00:00, 1556.88it/s]



统计结果 (单位: Tokens):
  平均长度: 320.3
  95% 覆盖: 497
  99% 覆盖: 563
  最大长度: 606
  建议值 (Max_Length): 640
正在分析文件: data/LogicalDeduction/dev.json (模式: Repeat)


...


统计结果 (单位: Tokens):
  平均长度: 410.0
  95% 覆盖: 526
  99% 覆盖: 534
  最大长度: 554
  建议值 (Max_Length): 640
正在分析文件: data/ProntoQA/dev.json (模式: Repeat)


100%|██████████| 500/500 [00:00<00:00, 1527.94it/s]



统计结果 (单位: Tokens):
  平均长度: 392.4
  95% 覆盖: 416
  99% 覆盖: 420
  最大长度: 428
  建议值 (Max_Length): 512
正在分析文件: data/ProofWriter/test.json (模式: Repeat)


100%|██████████| 600/600 [00:00<00:00, 1184.89it/s]


统计结果 (单位: Tokens):
  平均长度: 443.5
  95% 覆盖: 590
  99% 覆盖: 622
  最大长度: 642
  建议值 (Max_Length): 768
